# Hypoxia Focal Point Test (Criteria Filter)

This notebook identifies areas that meet the following three criteria:
1. **High Recent Hypoxia Frequency**: DO < 2.0 mg/L in >= 50% of the last 20 years (2004 - 2023).
2. **Highly Localized**: Decorrelation Scale < 10 km.
3. **Worsening Trend**: Hypoxia frequency in the recent decade (2014-2023) is higher than in the historical decade (1984-1993).

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Add project root to sys.path
current_dir = Path(os.getcwd()).resolve()
project_root = current_dir.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from utils.paths import ProjectPaths

# Apply project style
plt.style.use(str(ProjectPaths.PROJECT_ROOT / 'project_style.mplstyle'))

## 1. Data Loading

In [2]:
print("Loading data...")

# Load geometry data
merged_gdf = gpd.read_feather(ProjectPaths.MERGED_DATA_FILE)
stations_gdf = merged_gdf[['zettaicode', 'latitude', 'longitude', 'geometry', 'watermass_name']].drop_duplicates('zettaicode').set_index('zettaicode')

# Load Summer Bottom DO (yearly matrix)
subset_dir = ProjectPaths.get_subset_dir('Summer', 'bottom')
yearly_do_df = pd.read_csv(subset_dir / 'yearly_do_df.csv', index_col=0)
yearly_do_df.index = yearly_do_df.index.astype(str).str.zfill(8)

# Load Decorrelation Scale results
scale_path = subset_dir / 'stations_decorrelation_scale.csv'
if not scale_path.exists():
    raise FileNotFoundError(f"Decorrelation scale file not found: {scale_path}")
scale_df = pd.read_csv(scale_path, index_col=0)
if 'Station_ID' in scale_df.columns:
    scale_df['Station_ID'] = scale_df['Station_ID'].astype(str).str.zfill(8)
    scale_df = scale_df.set_index('Station_ID')

# Ensure indexes are strings
stations_gdf.index = stations_gdf.index.astype(str).str.zfill(8)

Loading data...


## 2. Criterion 1: Most Frequent in Recent Years
Criteria: Hypoxia (DO < 2.0) occurs in >= 50% of the valid years within the last 20 years (2004 - 2023).

In [3]:
HYPOXIA_THRESHOLD = 2.0
LAST_20_YEARS = [str(y) for y in range(2004, 2024)] 
# Ensure we only use years that exist in the dataframe
valid_recent_years = [y for y in LAST_20_YEARS if y in yearly_do_df.columns]

# Extract data for the last 20 years
do_recent_20 = yearly_do_df[valid_recent_years]

# Calculate frequency (ignoring NaNs)
hypoxia_count_20 = (do_recent_20 < HYPOXIA_THRESHOLD).sum(axis=1)
valid_years_20 = do_recent_20.notna().sum(axis=1)

# Frequency calculation
freq_recent_20 = hypoxia_count_20 / valid_years_20

# Filter
crit_1_stations = freq_recent_20[freq_recent_20 >= 0.5].index
print(f"Stations meeting Criterion 1 (>=50% frequency in last 20 years): {len(crit_1_stations)}")

Stations meeting Criterion 1 (>=50% frequency in last 20 years): 15


## 3. Criterion 2: Local Decorrelation Scale (< 10 km)

In [4]:
SCALE_THRESHOLD = 10.0

# Filter valid scales
valid_scale_df = scale_df[(scale_df['Decorrelation_Scale_km'] > 0) & (scale_df['Decorrelation_Scale_km'] < SCALE_THRESHOLD)]
crit_2_stations = valid_scale_df.index
print(f"Stations meeting Criterion 2 (Decorrelation Scale < {SCALE_THRESHOLD} km): {len(crit_2_stations)}")

Stations meeting Criterion 2 (Decorrelation Scale < 10.0 km): 95


## 4. Criterion 3: Worsening DO Trend (Notebook 42 Method)
Method: Proportional Delta ($\Delta = P_{recent} - P_{hist} > 0$).
We define historical as 1984-1993, and recent as 2014-2023.

In [5]:
HIST_YEARS = [str(y) for y in range(1984, 1994)]
RECENT_YEARS = [str(y) for y in range(2014, 2024)]

valid_hist = [y for y in HIST_YEARS if y in yearly_do_df.columns]
valid_recent = [y for y in RECENT_YEARS if y in yearly_do_df.columns]

p_hist = (yearly_do_df[valid_hist] < HYPOXIA_THRESHOLD).sum(axis=1) / 10
p_recent = (yearly_do_df[valid_recent] < HYPOXIA_THRESHOLD).sum(axis=1) / 10

delta = p_recent - p_hist

crit_3_stations = delta[delta > 0].index
print(f"Stations meeting Criterion 3 (Worsening trend, Delta > 0): {len(crit_3_stations)}")

Stations meeting Criterion 3 (Worsening trend, Delta > 0): 35


## 5. Intersection: The "Focal Points"
Find the stations that meet all three criteria.

In [6]:
# Intersect the sets
focal_points = set(crit_1_stations).intersection(set(crit_2_stations)).intersection(set(crit_3_stations))
focal_points = list(focal_points)
print(f"\nTotal Focal Point Stations found: {len(focal_points)}")

# Create a summary dataframe for the focal points
summary_data = []
for st in focal_points:
    summary_data.append({
        'Station_ID': st,
        'Region': stations_gdf.loc[st, 'watermass_name'],
        'Hypoxia_Freq_Last20yr': freq_recent_20.loc[st],
        'Decorrelation_Scale_km': scale_df.loc[st, 'Decorrelation_Scale_km'],
        'Trend_Delta': delta.loc[st],
        'Lat': stations_gdf.loc[st, 'latitude'],
        'Lon': stations_gdf.loc[st, 'longitude']
    })

if len(summary_data) > 0:
    focal_df = pd.DataFrame(summary_data)
    display(focal_df.sort_values(by='Hypoxia_Freq_Last20yr', ascending=False))
else:
    focal_df = pd.DataFrame()
    print("No stations met all criteria.")


Total Focal Point Stations found: 0
No stations met all criteria.


## 6. Mapping the Focal Points

In [7]:
if len(focal_df) > 0:
    fig = plt.figure(figsize=(15, 10))
    ax = plt.axes(projection=ccrs.PlateCarree())

    # Set extent for Seto Inland Sea
    ax.set_extent([130.6, 135.6, 32.75, 35.1], crs=ccrs.PlateCarree())

    # Add map features
    ax.coastlines(resolution='10m', color='black', linewidth=0.8)
    ax.add_feature(cfeature.LAND, facecolor='lightgray', edgecolor='black', linewidth=0.5)
    ax.add_feature(cfeature.OCEAN, facecolor='white')
    ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, linestyle=':', alpha=0.5)

    # Plot all valid stations (background)
    background_stations = stations_gdf[stations_gdf.index.isin(yearly_do_df.index)]
    ax.scatter(background_stations['longitude'], background_stations['latitude'], 
               color='lightgrey', s=10, alpha=0.5, transform=ccrs.PlateCarree(), label='Other Stations')

    # Plot Focal Points
    # Color by Delta, Size by Hypoxia Frequency
    sc = ax.scatter(focal_df['Lon'], focal_df['Lat'], 
                    c=focal_df['Trend_Delta'], cmap='Reds', 
                    s=focal_df['Hypoxia_Freq_Last20yr'] * 200, 
                    alpha=0.9, edgecolors='black', linewidth=1, 
                    transform=ccrs.PlateCarree(), label='Focal Points',
                    vmin=0, vmax=focal_df['Trend_Delta'].max())

    cbar = plt.colorbar(sc, ax=ax, orientation='vertical', pad=0.02, shrink=0.7)
    cbar.set_label('Worsening Trend (Delta)', fontsize=12)

    plt.title('Hypoxia Focal Points\n(Freq >= 50%, Scale < 10km, Worsening Trend)', fontsize=16, fontweight='bold', pad=20)
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()
else:
    print("No stations met all three criteria to plot.")

No stations met all three criteria to plot.
